# Discrete Bayesian Optimization for Ni-Cu Coking-Susceptibility Screening

This notebook reproduces the **exact BO workflow of Kayode, Hill & Montemore, *J. Mater. Chem. A*, 2023, 11, 19128** (the "single-atom alloy BO" paper), applied to your Ni-Cu bimetallic dataset instead of theirs. No machine-learned potential, relaxation, or continuous optimizer is used anywhere in the search loop — this is the paper's original **discrete-candidate-pool + Gaussian Process Regression (GPR) + Expected Improvement (EI)** workflow (their Fig. 1), reading pre-computed energies from a lookup table exactly as the paper does in most of its campaigns ("we performed the DFT calculations for the recommended SAA (or, in most cases, pulled them from the existing database)").

### Mapping paper → your system

| Paper (SAA screening) | This notebook (Ni-Cu bimetallic) |
|---|---|
| Search space = fixed set of SAA dopants | Search space = your 50 configurations (site + atomic arrangement) at fixed composition |
| Features: group number, period number, coupling element, **O$_{ads}$/C$_{ads}$** (cheap literature descriptor) | Features: **site-type dummy variables** (top/bridge/hollow) + **ML$_{Eads}$** (cheap FAIRChem-predicted energy, playing the same role as their O$_{ads}$/C$_{ads}$) |
| "DFT calculation" pulled from database | `ML+RX Eads` (full DFT relaxation) column, pulled from your table |
| Surrogate model: standard GPR | Same: `sklearn.GaussianProcessRegressor` |
| Acquisition: Expected Improvement, maximized over the *unexplored* discrete candidates | Same, adapted for target-value seeking (Section 3.1) |
| Stopping criteria: (1) recommendation within **±2%** of target, or (2) **14** additional DFT calls | Same, verbatim |
| Fig. 3: BO vs. random search, and feature ablation (with/without O$_{ads}$) across randomized targets | Reproduced here as validation, with/without ML$_{Eads}$ |

Two independent search campaigns are run — one per catalyst state (`fresh_catalyst`, 84:16 Ni:Cu; `regenerated_catalyst`, 77:23 Ni:Cu) — matching how the paper ran separate campaigns per host metal (Cu, Ag, Au) in Fig. 3b.


In [ ]:
import warnings
warnings.filterwarnings("ignore")  # suppress GPR hyperparameter-bound convergence chatter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

DATA_PATH = "dataset.xlsx"   # place dataset.xlsx alongside this notebook
np.random.seed(0)


## 1. Load the discrete candidate pool

Each row of your spreadsheet is one **discrete candidate**: a specific initial adsorption site (top/bridge/hollow) on a fixed-composition Ni-Cu slab. This is exactly the paper's "search space" — a finite, enumerable list of things BO can recommend for DFT, not a continuous surface to be optimized.

- `site`: adsorption site type → categorical descriptor (paper's HER dummy variables for site type, Section 3.4.3)
- `ML_Eads`: cheap FAIRChem-predicted adsorption energy → cheap descriptor (paper's O$_{ads}$/C$_{ads}$, Section 3.1/3.4.2)
- `Eads_DFT`: the **`ML+RX` full-relaxation DFT energy** → this is the "ground truth" value the workflow fetches whenever BO recommends a candidate (i.e., this plays the role of "performing the DFT calculation" in Fig. 1)


In [ ]:
def load_sheet(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet, header=[0, 1])
    df.columns = ["_".join([str(a), str(b)]).strip() for a, b in df.columns]

    rx_col = [c for c in df.columns if c.startswith("ML+RX_Eads")][0]
    sp_col = [c for c in df.columns if c.startswith("ML+SP_Eads")][0]

    out = pd.DataFrame({
        "site": df["ML_initial adsorption"],
        "ML_Eads": df["ML_Eads"],
        "Eads_DFT": df[rx_col],       # full-relaxation DFT -> ground truth "DFT calculation"
        "Eads_SP": df[sp_col],        # single-point DFT -> not used by default, available if wanted
    })
    out = out.dropna(subset=["site", "Eads_DFT"]).reset_index(drop=True)
    out["config_id"] = out.index + 1
    return out

fresh = load_sheet(DATA_PATH, "fresh_catalyst")
regen = load_sheet(DATA_PATH, "regenerated_catalyst")

datasets = {"fresh_catalyst (Ni84:Cu16)": fresh, "regenerated_catalyst (Ni77:Cu23)": regen}

for name, df in datasets.items():
    print(f"{name}: {len(df)} candidates, Eads range [{df.Eads_DFT.min():.3f}, {df.Eads_DFT.max():.3f}] eV")
    print(df["site"].value_counts().to_dict())


## 2. Feature set (paper's "simple, off-the-shelf features")

The paper is explicit that BO does not need high-level featurization — group number, period number, and a cheap reference adsorption energy were enough. Here:

- **Site dummy variables** — one-hot encoding of `top` / `bridge` / `hollow`, matching the dummy variables the paper adds for site type in their HER bimetallic screening (Section 3.4.3).
- **`ML_Eads`** — optional cheap descriptor, on/off exactly like the paper's O$_{ads}$/C$_{ads}$ ablation (Fig. 3d). Toggle with `use_ml_eads`.

No elemental composition features are included, since composition is **fixed** within each campaign (that's the whole point of running fresh/regenerated as separate campaigns, the same way the paper ran separate campaigns per host metal).


In [ ]:
def build_features(df, use_ml_eads=True):
    X = pd.get_dummies(df["site"], prefix="site").astype(float)
    if use_ml_eads:
        X["ML_Eads"] = df["ML_Eads"].values
    return X.values, list(X.columns)


## 3. Surrogate model and acquisition function (Fig. 1 of the paper)

**Surrogate:** standard GPR (Matern 5/2 kernel + noise), exactly as stated in Section 3.1 ("we used the standard Gaussian process regressor (GPR) as our surrogate model").

**Acquisition function:** the paper uses **expected improvement (EI)**, but their objective is not "maximize/minimize the property" — it's "get close to a **target value**" (Section 3.1: *"the goal of this workflow is to identify a SAA that has an adsorption energy that falls near a target value"*). Following the standard target-value-BO trick, we transform the objective to
$$g(x) = -\,|E_{ads}(x) - E_{target}|$$
and maximize expected improvement of $g$ (so "improvement" = "getting closer to target"). Because $|Y-\text{target}|$ of a Gaussian $Y$ has no simple closed form, EI is evaluated by fast Monte Carlo sampling of the GP posterior — this is a direct, transparent stand-in for the "contextual improvement" acquisition function the paper cites (ref. 43) for balancing exploration/exploitation.


In [ ]:
def make_gpr():
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5) \
             + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e1))
    return GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=3, random_state=0)


def expected_improvement_to_target(mu, sigma, target, best_g, n_mc=4000, rng=None):
    # Monte Carlo expected improvement toward a target value (paper Section 3.1).
    rng = rng or np.random.default_rng(0)
    sigma = np.maximum(sigma, 1e-8)
    samples = rng.normal(mu[:, None], sigma[:, None], size=(len(mu), n_mc))
    g_samples = -np.abs(samples - target)
    improvement = np.maximum(0.0, g_samples - best_g)
    return improvement.mean(axis=1)


## 4. The BO loop (paper's Fig. 1, verbatim structure)

1. Design search space and initial DFT set → **done above**
2. Construct GPR model on known points
3. Predict distribution/uncertainty over the remaining candidates
4. Compute the acquisition function; recommend the candidate with max EI
5. "Perform DFT" (look up `Eads_DFT` for the recommended candidate)
6. Check stopping criteria:
   - **(1)** recommendation's adsorption energy is within **±2%** of the target → stop, success
   - **(2)** **14** additional DFT calculations reached → stop, arbitrary cutoff (paper's exact threshold)
7. If neither met, add the recommendation to the known set and repeat from step 2


In [ ]:
def run_bo_campaign(df, X, target, init_idx, max_calls=14, tol_frac=0.02, rng=None, verbose=False):
    rng = rng or np.random.default_rng(0)
    y = df["Eads_DFT"].values
    n = len(df)
    known = list(init_idx)
    unknown = [i for i in range(n) if i not in known]
    tol = tol_frac * abs(target)

    # Check whether the initial seed set already satisfies the target (as in the paper's Fig. 2 example)
    for i in known:
        if abs(y[i] - target) <= tol:
            return 0, [], i

    n_calls = 0
    history = []
    while unknown and n_calls < max_calls:
        gpr = make_gpr()
        gpr.fit(X[known], y[known])
        mu, sigma = gpr.predict(X[unknown], return_std=True)

        g_known = -np.abs(y[known] - target)
        best_g = g_known.max()
        ei = expected_improvement_to_target(mu, sigma, target, best_g, rng=rng)

        pick_local = int(np.argmax(ei))
        pick = unknown[pick_local]
        known.append(pick)
        unknown.remove(pick)
        n_calls += 1
        history.append(pick)

        if verbose:
            print(f"  recommendation {n_calls}: config {df.loc[pick,'config_id']} "
                  f"({df.loc[pick,'site']}), Eads_DFT = {y[pick]:.3f} eV")

        if abs(y[pick] - target) <= tol:
            return n_calls, history, pick

    return n_calls, history, None  # stopping criterion 2 reached, no candidate within tolerance


def run_random_campaign(df, target, init_idx, max_calls=14, tol_frac=0.02, rng=None):
    # Random-search baseline (paper Fig. 3c): draw recommendations uniformly at random.
    rng = rng or np.random.default_rng(0)
    y = df["Eads_DFT"].values
    n = len(df)
    known = list(init_idx)
    unknown = [i for i in range(n) if i not in known]
    tol = tol_frac * abs(target)

    for i in known:
        if abs(y[i] - target) <= tol:
            return 0

    order = list(unknown)
    rng.shuffle(order)
    n_calls = 0
    for pick in order:
        n_calls += 1
        if abs(y[pick] - target) <= tol or n_calls >= max_calls:
            return n_calls
    return n_calls


## 5. Worked single-target example (reproduces the style of paper's Fig. 2)

Set `TARGET_EADS` below to whatever adsorption energy you actually want a Ni-Cu configuration for (your DFT range is roughly **-6.5 to -5.1 eV**). By default this uses **8 initial seed points** (identical to the paper's Fig. 2 example), chosen to sample across the three site types the way the paper hand-picks its initial set "to sample across the periodic table."


In [ ]:
TARGET_EADS = -5.8     # <-- edit this to your actual target adsorption energy
CATALYST = "fresh_catalyst (Ni84:Cu16)"   # or "regenerated_catalyst (Ni77:Cu23)"

df = datasets[CATALYST]
X, feat_names = build_features(df, use_ml_eads=True)

# stratified initial seed: sample across all site types, 8 total (paper's Fig. 2 seed size)
rng = np.random.default_rng(0)
init_idx = []
sites = df["site"].unique()
per_site = 8 // len(sites)
for s in sites:
    idx_s = df.index[df["site"] == s].tolist()
    init_idx += list(rng.choice(idx_s, size=min(per_site, len(idx_s)), replace=False))
while len(init_idx) < 8:
    cand = rng.choice(df.index)
    if cand not in init_idx:
        init_idx.append(cand)

print(f"Target: {TARGET_EADS} eV  |  Catalyst: {CATALYST}")
print(f"Initial seed set ({len(init_idx)} candidates): configs "
      f"{df.loc[init_idx, 'config_id'].tolist()}")

n_calls, history, hit = run_bo_campaign(df, X, TARGET_EADS, init_idx, rng=rng, verbose=True)

if hit is not None:
    print(f"\nStopping criterion 1 met: config {df.loc[hit,'config_id']} "
          f"({df.loc[hit,'site']}) has Eads_DFT = {df.loc[hit,'Eads_DFT']:.3f} eV "
          f"after {n_calls} additional DFT calculation(s).")
else:
    print(f"\nStopping criterion 2 met: no candidate within \u00b12% of target after {n_calls} DFT calls.")


In [ ]:
# Fig. 2-style plot: adsorption energy of each recommendation vs. number of DFT calculations
fig, ax = plt.subplots(figsize=(5, 4))
vals = [df.loc[i, "Eads_DFT"] for i in history]
ax.plot(range(1, len(vals) + 1), vals, "o--", color="tab:red", alpha=0.6)
ax.axhline(TARGET_EADS, color="k", ls="--", label="Target")
ax.axhspan(TARGET_EADS * 1.02 if TARGET_EADS < 0 else TARGET_EADS * 0.98,
           TARGET_EADS * 0.98 if TARGET_EADS < 0 else TARGET_EADS * 1.02,
           color="gray", alpha=0.15, label="\u00b12% window")
for i, (x, y) in enumerate(zip(range(1, len(vals) + 1), vals)):
    ax.annotate(f"{df.loc[history[i],'config_id']}", (x, y), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("No. of DFT calculations")
ax.set_ylabel("Adsorption energy (eV)")
ax.set_title(f"BO search campaign \u2014 {CATALYST}")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Validation: BO vs. random search, across randomized targets (reproduces Fig. 3c)

The paper validates its workflow by generating **randomly chosen target energies** within the range of the dataset and running many independent campaigns, then comparing the average number of additional DFT calculations needed for BO vs. a random-search baseline. We do the same here for each catalyst state.


In [ ]:
N_TRIALS = 25
N_INIT = 8

def validation_campaign(df, use_ml_eads=True, n_trials=N_TRIALS, n_init=N_INIT, seed=0):
    X, _ = build_features(df, use_ml_eads=use_ml_eads)
    rng_master = np.random.default_rng(seed)
    lo, hi = df["Eads_DFT"].min() + 0.05, df["Eads_DFT"].max() - 0.05
    targets = rng_master.uniform(lo, hi, size=n_trials)

    bo_calls, rand_calls = [], []
    for t in targets:
        trial_rng = np.random.default_rng(rng_master.integers(0, 1_000_000))
        init_idx = list(trial_rng.choice(len(df), size=n_init, replace=False))
        nc_bo, _, _ = run_bo_campaign(df, X, t, init_idx, rng=trial_rng)
        nc_rand = run_random_campaign(df, t, init_idx, rng=trial_rng)
        bo_calls.append(nc_bo)
        rand_calls.append(nc_rand)
    return np.array(targets), np.array(bo_calls), np.array(rand_calls)


results = {}
for name, d in datasets.items():
    targets, bo_calls, rand_calls = validation_campaign(d)
    results[name] = (targets, bo_calls, rand_calls)
    print(f"{name}: BO mean = {bo_calls.mean():.2f} \u00b1 {bo_calls.std():.2f} DFT calls, "
          f"Random mean = {rand_calls.mean():.2f} \u00b1 {rand_calls.std():.2f} DFT calls "
          f"({n_trials if (n_trials:=N_TRIALS) else ''} trials)")


In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4), sharey=True)
if len(results) == 1:
    axes = [axes]
for ax, (name, (targets, bo_calls, rand_calls)) in zip(axes, results.items()):
    means = [bo_calls.mean(), rand_calls.mean()]
    stds = [bo_calls.std(), rand_calls.std()]
    ax.bar(["Bayesian\noptimization", "Random\nsearch"], means, yerr=stds,
           color=["tab:green", "khaki"], capsize=5)
    ax.set_title(name)
    ax.set_ylabel("No. of DFT calculations")
plt.tight_layout()
plt.show()


## 7. Feature ablation: with vs. without `ML_Eads` (reproduces Fig. 3d)

The paper tests how much their cheap O$_{ads}$/C$_{ads}$ descriptor helps search efficiency by running the same campaigns with and without it. Here we do the identical test for `ML_Eads`.


In [ ]:
ablation_results = {}
for name, d in datasets.items():
    _, bo_calls_with, _ = validation_campaign(d, use_ml_eads=True, seed=1)
    _, bo_calls_without, _ = validation_campaign(d, use_ml_eads=False, seed=1)
    ablation_results[name] = (bo_calls_with, bo_calls_without)
    print(f"{name}: with ML_Eads = {bo_calls_with.mean():.2f} \u00b1 {bo_calls_with.std():.2f}, "
          f"without ML_Eads = {bo_calls_without.mean():.2f} \u00b1 {bo_calls_without.std():.2f}")


In [ ]:
fig, axes = plt.subplots(1, len(ablation_results), figsize=(5 * len(ablation_results), 4), sharey=True)
if len(ablation_results) == 1:
    axes = [axes]
for ax, (name, (with_ml, without_ml)) in zip(axes, ablation_results.items()):
    means = [with_ml.mean(), without_ml.mean()]
    stds = [with_ml.std(), without_ml.std()]
    ax.bar(["With ML_Eads\nfeature", "Without ML_Eads\nfeature"], means, yerr=stds,
           color=["steelblue", "sandybrown"], capsize=5)
    ax.set_title(name)
    ax.set_ylabel("No. of DFT calculations")
plt.tight_layout()
plt.show()


## Notes / how to adapt this further

- **`TARGET_EADS`** in Section 5 is the only thing you *must* set for a real single-target search — pick it based on the specific adsorption strength you're screening for (e.g., a value on the weak-adsorption side to identify configurations that resist $CH_3$/coking-agent binding, consistent with your Ni-Cu design rationale).
- To search a **new, unmeasured** configuration instead of pulling `Eads_DFT` from the table, replace the lookup inside `run_bo_campaign` (the line reading `y[pick]`) with an actual DFT/VASP submission for the recommended `config_id`, then feed the resulting energy back in — this is exactly how the paper's workflow operates when it isn't drawing from an existing database (Section 3.4.2, CO$_2$ reduction case).
- `max_calls=14` and `tol_frac=0.02` are the paper's literal values (Section 3.1); change only if you have a reason to.
- If you want `ML+SP` (single-point DFT) treated as a *second* cheap descriptor rather than `ML_Eads`, swap it in inside `build_features` — the ablation logic is identical either way.
